In [4]:
!pip install xlsxwriter

In [6]:
!pip install openpyxl

In [2]:
import sys
!{sys.executable} -m pip install openpyxl

In [ ]:
import openpyxl
import os

# 1. Define the list of filenames
target_files = [
    "hosp-epis-stat-admi-diag-2018-19-tab.xlsx",
    "hosp-epis-stat-admi-diag-2019-20-tab supp.xlsx",
    "hosp-epis-stat-admi-diag-2020-21-tab.xlsx",
    "hosp-epis-stat-admi-diag-2021-22-tab.xlsx"
]

# 2. Define target worksheets to process
target_sheets = ["Primary Diagnosis Summary", "Primary Diagnosis 3 Character"]

# 3. Define ICD gender logic rules (first three digits)
MALE_ONLY = [f'C{i}' for i in range(60, 64)] + [f'N{i}' for i in range(40, 52)]
FEMALE_ONLY = ([f'C{i}' for i in range(51, 59)] + 
               [f'N{i}' for i in range(70, 78)] + 
               [f'N{i}' for i in range(80, 99)] + 
               [f'O{i:02d}' for i in range(0, 100)])

def safe_int(val):
    """Safely convert to integer; returns 0 if '*' or empty/blank"""
    try:
        return int(val)
    except (ValueError, TypeError):
        return 0

def process_inplace_reclassify(file_path):
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return

    print(f"Opening and modifying: {file_path}")
    wb = openpyxl.load_workbook(file_path)
    
    for sheet_name in target_sheets:
        if sheet_name not in wb.sheetnames:
            continue
            
        ws = wb[sheet_name]
        
        # Based on screenshots, headers are in row 11
        header_row = 11
        
        col_code_idx, col_desc_idx = None, None
        col_male_idx, col_female_idx, col_unknown_idx = None, None, None
        
        # Identify column positions dynamically
        for cell in ws[header_row]:
            # Convert to lowercase for matching to increase robustness
            val = str(cell.value).strip().lower() if cell.value else ""
            
            if 'primary diagnosis' in val or 'icd-10' in val:
                col_code_idx = cell.column
                col_desc_idx = cell.column + 1 # Default description is to the right of the code
            elif val == 'male':
                col_male_idx = cell.column
            elif val == 'female':
                col_female_idx = cell.column
            elif val == 'gender unknown':
                col_unknown_idx = cell.column
                
        # If the code column is still not found, default to columns 1 and 2
        if not col_code_idx: 
            col_code_idx, col_desc_idx = 1, 2
            
        if not all([col_male_idx, col_female_idx, col_unknown_idx]):
            print(f"  - [{sheet_name}] Warning: Could not find Male/Female/Gender Unknown columns. Skipping sheet.")
            continue

        # Iterate through data starting from row 12
        for r in range(12, ws.max_row + 1):
            code_cell = ws.cell(row=r, column=col_code_idx)
            desc_cell = ws.cell(row=r, column=col_desc_idx)
            
            raw_code = str(code_cell.value) if code_cell.value else ""
            raw_desc = str(desc_cell.value) if desc_cell.value else ""
            
            # --- 1. Remove the ‡ symbol ---
            if '‡' in raw_code:
                code_cell.value = raw_code.replace('‡', '').strip()
            if '‡' in raw_desc:
                desc_cell.value = raw_desc.replace('‡', '').strip()
                
            clean_code = str(code_cell.value).strip()
            prefix = clean_code[:3]
            
            # --- 2. Data reclassification logic ---
            male_cell = ws.cell(row=r, column=col_male_idx)
            female_cell = ws.cell(row=r, column=col_female_idx)
            unknown_cell = ws.cell(row=r, column=col_unknown_idx)
            
            if prefix in MALE_ONLY:
                f_val = safe_int(female_cell.value)
                if f_val > 0:
                    current_unknown = safe_int(unknown_cell.value)
                    # Move incorrect female counts to Gender Unknown
                    unknown_cell.value = current_unknown + f_val 
                    # Set original female column to 0
                    female_cell.value = 0                        
                    
            elif prefix in FEMALE_ONLY:
                m_val = safe_int(male_cell.value)
                if m_val > 0:
                    current_unknown = safe_int(unknown_cell.value)
                    # Move incorrect male counts to Gender Unknown
                    unknown_cell.value = current_unknown + m_val 
                    # Set original male column to 0
                    male_cell.value = 0                          
                    
        print(f"  - [{sheet_name}] Symbol cleaning and gender reclassification complete.")

    # Overwrite the original file
    wb.save(file_path)
    wb.close()
    print(f"Changes saved to original file: {file_path}\n")

# Execute processing
print("Starting in-place data modification...\n")
for file in target_files:
    process_inplace_reclassify(file)
print("All tasks completed! You can now open the Excel files to check the results.")

In [ ]:
import openpyxl
import os

# 1. Target files for 2020-22
target_files = [
    "hosp-epis-stat-admi-diag-2020-21-tab.xlsx",
    "hosp-epis-stat-admi-diag-2021-22-tab.xlsx"
]

# 2. Target sheet name
target_sheet = "Primary Diagnosis Summary"

# 3. Define ICD gender logic rules (First three characters)
MALE_ONLY = [f'C{i}' for i in range(60, 64)] + [f'N{i}' for i in range(40, 52)]
FEMALE_ONLY = ([f'C{i}' for i in range(51, 59)] + 
               [f'N{i}' for i in range(70, 78)] + 
               [f'N{i}' for i in range(80, 99)] + 
               [f'O{i:02d}' for i in range(0, 100)])

def safe_int(val):
    """Safely convert value to integer"""
    try:
        return int(val)
    except (ValueError, TypeError):
        return 0

def process_summary_hardcoded(file_path):
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return

    print(f"Opening and modifying: {file_path}")
    wb = openpyxl.load_workbook(file_path)
    
    if target_sheet not in wb.sheetnames:
        print(f"  - Error: Cannot find sheet {target_sheet}")
        return
        
    ws = wb[target_sheet]
    
    # Fixed column indices (A=1, B=2 ... E=5, F=6, G=7)
    col_code_idx = 1     # Column A: Contains ‡ symbol, ICD code, and description
    col_male_idx = 5     # Column E: Male
    col_female_idx = 6   # Column F: Female
    col_unknown_idx = 7  # Column G: Gender Unknown

    # Iterate from row 11 to avoid complex headers
    for r in range(11, ws.max_row + 1):
        code_cell = ws.cell(row=r, column=col_code_idx)
        
        # Skip if column A is empty
        if not code_cell.value:
            continue
            
        raw_code = str(code_cell.value)
        
        # --- 1. Remove ‡ symbol ---
        if '‡' in raw_code:
            raw_code = raw_code.replace('‡', '').strip()
            code_cell.value = raw_code  # Write back cleaned text
            
        # Extract first 3 characters for logic validation
        clean_code = raw_code.strip()
        prefix = clean_code[:3]
        
        # --- 2. Data reclassification logic ---
        male_cell = ws.cell(row=r, column=col_male_idx)
        female_cell = ws.cell(row=r, column=col_female_idx)
        unknown_cell = ws.cell(row=r, column=col_unknown_idx)
        
        if prefix in MALE_ONLY:
            f_val = safe_int(female_cell.value)
            if f_val > 0:
                current_unknown = safe_int(unknown_cell.value)
                unknown_cell.value = current_unknown + f_val # Transfer value
                female_cell.value = 0                        # Clear original cell
                
        elif prefix in FEMALE_ONLY:
            m_val = safe_int(male_cell.value)
            if m_val > 0:
                current_unknown = safe_int(unknown_cell.value)
                unknown_cell.value = current_unknown + m_val # Transfer value
                male_cell.value = 0                          # Clear original cell
                
    print(f"  - [{target_sheet}] Symbol cleaning and gender reclassification completed.")

    # Save changes to the original file
    wb.save(file_path)
    wb.close()
    print(f"Changes saved to: {file_path}\n")

# Execution
print("Starting Summary sheet corrections...\n")
for file in target_files:
    process_summary_hardcoded(file)
print("All tasks completed!")